# Sentiment Analysis — Merdeka Curriculum Discourse on X

Portfolio version of the sentiment-analysis pipeline used in the published study **“Sentiment and Social Network Analysis of X Social Media on the Implementation of the Merdeka Belajar Curriculum in Indonesia”** (Journal of Advanced Technology and Multidiscipline, 2025).

This notebook focuses on the **sentiment-analysis branch** of the study. The published research analyzed public discourse collected from X and compared **SVM, Random Forest, XGBoost, and LightGBM** using TF-IDF features and SMOTE-based class balancing.

> **Portfolio note:** Raw social-media data, usernames, tweet URLs, local file paths, and ad-hoc experimental cells are excluded from this public version. The notebook preserves the final modeling approach reported in the publication.

## Published Model Comparison

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 |
|---|---:|---:|---:|---:|
| SVM | 0.65 | 0.51 | 0.55 | 0.52 |
| **Random Forest** | **0.70** | **0.59** | 0.52 | **0.53** |
| XGBoost | 0.68 | 0.53 | 0.51 | 0.52 |
| LightGBM | 0.68 | 0.53 | **0.54** | **0.53** |

**Random Forest** achieved the highest accuracy in the published comparison.

---


## **Sentiment Analysis Merdeka Belajar Curriculum**

### **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from emoji import demojize
import string
from gensim.models import Word2Vec
import emoji
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek
from sklearn.linear_model import LogisticRegression
from nltk.stem import WordNetLemmatizer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from wordcloud import WordCloud
import warnings
from collections import Counter
warnings.filterwarnings('ignore')
nltk.download('punkt')
nltk.download('stopwords')
from sklearn.naive_bayes import MultinomialNB

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as IMBPipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb

### **Import Dataset**

In [ ]:
df = pd.read_csv('path/to/cleaned_labeled_tweets.csv')
df.head()

### **Data Understanding**

In [ ]:
df.info()

In [ ]:
df['label'].value_counts()

In [ ]:
df['cleaned_text'].astype('str')

### **EDA**

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = ' '.join(df['full_text'])
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud Sebelum Prepocessing Data', fontsize=15)
plt.show()

In [ ]:
text = ' '.join(df['full_text'])
words = text.split()
word_counts = Counter(words)
most_common = word_counts.most_common(10)
words, counts = zip(*most_common)
plt.figure(figsize=(10, 6))
plt.bar(words, counts, color='skyblue')
plt.title('10 Kata Paling Sering Muncul', fontsize=15)
plt.xlabel('Kata', fontsize=12)
plt.ylabel('Frekuensi', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_cleaned = df.dropna(subset=['cleaned_text'])

In [ ]:
text = ' '.join(df_cleaned['cleaned_text'])
words = text.split()
word_counts = Counter(words)
most_common = word_counts.most_common(10)
words, counts = zip(*most_common)
plt.figure(figsize=(10, 6))
plt.bar(words, counts, color='skyblue')
plt.title('10 Kata Paling Sering Muncul', fontsize=15)
plt.xlabel('Kata', fontsize=12)
plt.ylabel('Frekuensi', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
text = ' '.join(df['full_text'])
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud Sebelum Prepocessing Data', fontsize=15)
plt.show()

In [ ]:
counts = df['label'].value_counts()

# Mapping label untuk kategori
label_mapping = {0: "Netral", 1: "Positif", 2: "Negatif"}
counts.index = [label_mapping[idx] for idx in counts.index]  # Ubah index menjadi nama kategori

# Membuat plot
plt.figure(figsize=(8, 6))
bars = counts.plot(kind='bar', color=['lightblue', 'lightgreen', 'salmon'], legend=False)

# Menambahkan angka di atas bar
for bar in bars.patches:
    plt.text(
        bar.get_x() + bar.get_width() / 2,  # Koordinat X
        bar.get_height() + 0.5,  # Koordinat Y (di atas bar)
        int(bar.get_height()),  # Teks (jumlah)
        ha='center', va='bottom', fontsize=10
    )

# Menambahkan judul dan label sumbu
plt.title('Distribusi Label', fontsize=14)
plt.xlabel('Kategori', fontsize=12)
plt.ylabel('Jumlah', fontsize=12)
plt.xticks(rotation=0, fontsize=11)
plt.show()

In [ ]:
# Remove rows where 'cleaned_text' is empty (NaN)
df_cleaned = df.dropna(subset=['cleaned_text'])

In [ ]:
if 'cleaned_text' in df.columns:
    from wordcloud import WordCloud

    all_processed_text = ' '.join(df_cleaned['cleaned_text'].tolist())
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_processed_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  
    plt.title('Word Cloud of Processed Text')
    plt.show()

else:
    print("The cleaned_text column is missing.")

In [ ]:
df_cleaned['label'].value_counts()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Mapping label angka ke nama sentimen
sentiment_map = {0: "Neutral", 1: "Positive", 2: "Negative"}

# Function to generate word cloud for each sentiment
def generate_wordcloud(data, sentiment):
    text = ' '.join(data[data['label'] == sentiment]['cleaned_text'])

    # Generate WordCloud tanpa menghapus kata-kunci tertentu
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    
    # Get sentiment name from mapping
    sentiment_name = sentiment_map.get(sentiment, sentiment)

    # Display the generated image
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for {sentiment_name} Sentiment')
    plt.show()

# Get unique sentiments in the training data
sentiments = df_cleaned['label'].unique()

# Generate and display word clouds for each sentiment
for sentiment in sentiments:
    generate_wordcloud(df_cleaned, sentiment)


In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Daftar kata yang ingin dihapus
custom_stopwords = {"kurikulum", "merdeka", "belajar", "nadiem", "kurikulum merdeka", "merdeka belajar", "kurikulum nadiem"}

# Mapping label angka ke nama sentimen
sentiment_map = {0: "Neutral", 1: "Positive", 2: "Negative"}

# Function to generate word cloud for each sentiment
def generate_wordcloud(data, sentiment):
    text = ' '.join(data[data['label'] == sentiment]['cleaned_text'])
    
    # Update stopwords dengan kata-kata yang ingin dihapus
    stopwords = STOPWORDS.union(custom_stopwords)

    # Generate WordCloud
    wordcloud = WordCloud(width=800, height=400, background_color='white', stopwords=stopwords).generate(text)
    
    # Get sentiment name from mapping
    sentiment_name = sentiment_map.get(sentiment, sentiment)

    # Display the generated image
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for {sentiment_name} Sentiment')
    plt.show()

# Get unique sentiments in the training data
sentiments = df_cleaned['label'].unique()

# Generate and display word clouds for each sentiment
for sentiment in sentiments:
    generate_wordcloud(df_cleaned, sentiment)


### **Splitting Data**

In [ ]:
X = df_cleaned['cleaned_text']
y = df_cleaned['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  

### **Modelling**

In [ ]:
svm_model = LinearSVC()
rf_model = RandomForestClassifier(random_state=42)
xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
lgbm_model = lgb.LGBMClassifier(random_state=42)

In [ ]:
# Inisialisasi TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Inisialisasi SMOTE untuk menangani class imbalance
smote = SMOTE(random_state=42)

In [ ]:
pipeline_svm = IMBPipeline([('tfidf', tfidf_vectorizer), ('smote', smote), ('svm', svm_model)])
pipeline_rf = IMBPipeline([('tfidf', tfidf_vectorizer), ('smote', smote), ('rf', rf_model)])
pipeline_xgb = IMBPipeline([('tfidf', tfidf_vectorizer), ('smote', smote), ('xgb', xgb_model)])
pipeline_lgbm = IMBPipeline([('tfidf', tfidf_vectorizer), ('smote', smote), ('lgbm', lgbm_model)])

In [ ]:
pipeline_svm.fit(X_train, y_train)
pipeline_rf.fit(X_train, y_train)
pipeline_xgb.fit(X_train, y_train)
pipeline_lgbm.fit(X_train, y_train)

In [ ]:
y_pred_svm = pipeline_svm.predict(X_test)
y_pred_rf = pipeline_rf.predict(X_test)
y_pred_xgb = pipeline_xgb.predict(X_test)
y_pred_lgbm = pipeline_lgbm.predict(X_test)

In [ ]:
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

In [ ]:
# Function to plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Netral", "Positive", "Negative"], yticklabels=["Netral", "Positive", "Negative"])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# Confusion Matrix for each model
plot_confusion_matrix(y_test, y_pred_svm, "SVM Confusion Matrix")
plot_confusion_matrix(y_test, y_pred_rf, "Random Forest Confusion Matrix")
plot_confusion_matrix(y_test, y_pred_xgb, "XGBoost Confusion Matrix")
plot_confusion_matrix(y_test, y_pred_lgbm, "LightGBM Confusion Matrix")

In [ ]:
# Get the misclassified indices
misclassified_mask = y_pred_rf != y_test

# Get the misclassified data based on the mask
misclassified_examples = pd.DataFrame({
    'cleaned_text': X_test[misclassified_mask],
    'true_label': y_test[misclassified_mask],
    'predicted_label': y_pred_rf[misclassified_mask]
})

pd.set_option('display.max_colwidth', None) 
              
# Display the misclassified data
print("Misclassified Data:")
misclassified_examples

# Optional: To view a portion of the misclassified data
misclassified_examples.head()

In [ ]:
# Filter misclassified data where true label is 1 and predicted is 2
specific_misclassified = misclassified_examples[
    (misclassified_examples['true_label'] == 1) & 
    (misclassified_examples['predicted_label'] == 2)
]

# Tampilkan hasilnya
print("True label = 1, Predicted label = 2:")
specific_misclassified.tail(60)  # atau .to_string(index=False) kalau mau rapi